# 01 — Inspeção das fontes (dias 1–2)

Objetivo: olhar os dados brutos de cada fonte e responder ao **portão de decisão** (briefing, seção 13).

Antes, rode na raiz do projeto:
```bash
make checar-fontes
python -m src.collect.cge_alagamentos baixar  --inicio 2024-01-01 --fim 2024-01-31
python -m src.collect.cge_alagamentos extrair --inicio 2024-01-01 --fim 2024-01-31
python -m src.collect.geosampa_wfs camadas --filtro alag
python -m src.collect.geosampa_wfs baixar --camada geoportal:<camada>
python -m src.collect.geosampa_wfs extrair --arquivo data/raw/geosampa/<camada>.geojson
```

In [ ]:
# Preparação: permite importar src/ a partir da pasta notebooks/
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config

pd.set_option("display.max_columns", 50)
print("Raiz do projeto:", RAIZ)

## 1. CGE — alagamentos (1 mês de teste)
Confira algumas linhas contra o site. Colunas vazias indicam seletor/regex a ajustar em `src/collect/cge_alagamentos.py`.

In [ ]:
cge = pd.read_parquet(config.ARQ_EVENTOS_CGE)
cge.info()
display(cge.head(15))
print("\nFração de vazios por coluna:")
print(cge.isna().mean().round(3))

In [ ]:
# Distribuição do horário de início (hora local) e duração
inicio = pd.to_datetime(cge["inicio_local"])
fim = pd.to_datetime(cge["fim_local"])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
inicio.dt.hour.value_counts().sort_index().plot.bar(ax=ax[0], title="Início por hora do dia")
((fim - inicio).dt.total_seconds() / 60).plot.hist(bins=30, ax=ax[1], title="Duração (min)")
plt.tight_layout()
print(cge["transitavel"].value_counts(dropna=False))

## 2. GeoSampa — ocorrências de alagamento
Leia o **Dicionário de Dados** antes de interpretar as colunas.

In [ ]:
geo = pd.read_parquet(config.ARQ_EVENTOS_GEOSAMPA)
geo.info()
display(geo.head(10))
geo["ano"] = pd.to_datetime(geo["inicio_local"]).dt.year
print(geo["ano"].value_counts().sort_index())

In [ ]:
# Hora do registro: se houver pico em horário comercial, é sinal de que é a hora do chamado
pd.to_datetime(geo["inicio_local"]).dt.hour.value_counts().sort_index().plot.bar(figsize=(8, 3), title="GeoSampa: hora do registro")

## 3. Open-Meteo — histórico × ao vivo
A mesma hora deve ter valores parecidos nas três fontes (a reanálise pode ter alguns dias de atraso).

In [ ]:
from src.collect.open_meteo import comparar_historico_e_ao_vivo, buscar_historico

comparacao = comparar_historico_e_ao_vivo(-23.5152, -46.5841, dias=3)
display(comparacao.dropna(how="all").tail(48))
print(comparacao.sum())

In [ ]:
# Cobertura da API de previsões históricas (decide a recomendação R1)
for ano in (2016, 2018, 2020, 2021, 2022, 2023):
    try:
        s = buscar_historico([(-23.5152, -46.5841)], f"{ano}-01-01", f"{ano}-01-10",
                             fonte="previsoes_historicas")[0].serie
        print(ano, "horas com valor:", int(s.notna().sum()), "| soma:", round(float(s.sum()), 1))
    except Exception as erro:
        print(ano, "ERRO:", erro)

In [ ]:
# Jan–mar/2024 na Vila Maria: arquivo × previsões históricas
arq = buscar_historico([(-23.5152, -46.5841)], "2024-01-01", "2024-03-31", fonte="arquivo")[0].serie
prev = buscar_historico([(-23.5152, -46.5841)], "2024-01-01", "2024-03-31", fonte="previsoes_historicas")[0].serie
duas = pd.DataFrame({"arquivo": arq, "previsoes_historicas": prev})
print(duas.describe())
print("correlação horária:", duas.corr().iloc[0, 1].round(3))
duas.resample("D").sum().plot(figsize=(11, 3), title="Chuva diária — Vila Maria")

## 4. Boletim pluviométrico do CGE (melhoria)

In [ ]:
from src.collect.cge_pluvio import DIR_PLUVIO, ler_arquivo

arquivos = sorted(DIR_PLUVIO.glob("*.*"))
print(arquivos)
if arquivos:
    bruto = ler_arquivo(arquivos[0])
    display(bruto.head(20))
    print(bruto.columns.tolist())

## 5. Portão de decisão — respostas

| Pergunta | Resposta (com número) |
|---|---|
| CGE tem hora de início e dá para raspar vários anos? | |
| Open-Meteo histórico bate com o ao vivo? Cobertura das previsões históricas? | |
| Positivos suficientes (cidade / ZN)? | |
| Plano escolhido (principal / B / C) | |

Copie as respostas para `DATASET.md` e `docs/coleta-dados.md`.